# SI630 Final Project

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import TweetTokenizer
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

import wandb
from tqdm import tqdm

from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
    BertModel
)
import sys, subprocess
import transformers
import accelerate
import json
wandb.login()

e:\OneDrive - Umich\Umich\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\23639\_netrc.
wandb: Currently logged in as: ffliu (ffliu-university-of-michigan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## import Dataset & Data Cleaning & Visualization

In [ ]:
ROOT = Path.cwd()
DATA_DIR = ROOT / "sarcasm_v2"

files = {
    "GEN": DATA_DIR / "GEN-sarc-notsarc.csv",
    "HYP": DATA_DIR / "HYP-sarc-notsarc.csv",
    "RQ": DATA_DIR / "RQ-sarc-notsarc.csv",
}

frames: dict[str, pd.DataFrame] = {}
for name, path in files.items():
    frames[name] = pd.read_csv(path)
GEN = frames["GEN"]
HYP = frames["HYP"]
RQ = frames["RQ"]

# Combine into one DataFrame
all_df = pd.concat(
    [
        GEN.assign(source="GEN"),
        HYP.assign(source="HYP"),
        RQ.assign(source="RQ"),
    ],
    ignore_index=True,
)

print("Combined:", all_df.shape)
display(all_df.sample(5))

Combined: (9386, 4)


,class,id,text,source
6073,sarc,6074,You never bought something you liked? You neve...,GEN
3602,sarc,3603,"I have no idea what ""women's rights"" might be ...",GEN
9268,sarc,1585,ever read job chapter 1 (etc.) in the tanakh? ...,RQ
5078,sarc,5079,"So, if having children isn't a requirement fro...",GEN
3090,sarc,3091,"Ahh, pointy tailed devils with pitchforks, mor...",GEN


In [3]:
def class_dist(df: pd.DataFrame, label_col: str = "class") -> pd.DataFrame:
    counts = df[label_col].value_counts(dropna=False)
    perc = (counts / len(df) * 100).round(2)
    out = pd.DataFrame({"count": counts, "percent": perc})
    return out

for name, df_ in frames.items():
    print(f"\n{name} class distribution:")
    display(class_dist(df_))

print("\nALL class distribution:")
display(class_dist(all_df))


GEN class distribution:


,count,percent
class,,
notsarc,3260,50.0
sarc,3260,50.0



HYP class distribution:


,count,percent
class,,
notsarc,582,50.0
sarc,582,50.0



RQ class distribution:


,count,percent
class,,
notsarc,851,50.0
sarc,851,50.0



ALL class distribution:


,count,percent
class,,
notsarc,4693,50.0
sarc,4693,50.0


## Tokenization

In [4]:
tokenizer = TweetTokenizer(preserve_case = True)
all_df['text'] = all_df['text'].fillna("").astype(str)
all_df['tokens'] = all_df['text'].apply(tokenizer.tokenize)
print(all_df[['text', 'tokens']].head(3))

                                                text  \
0  If that's true, then Freedom of Speech is doom...   
1  Neener neener - is it time to go in from the p...   
2  Just like the plastic gun fear, the armour pie...   

                                              tokens  
0  [If, that's, true, ,, then, Freedom, of, Speec...  
1  [Neener, neener, -, is, it, time, to, go, in, ...  
2  [Just, like, the, plastic, gun, fear, ,, the, ...  


## Baseline tests

In [ ]:
# Baseline 1: Random-choice classifier
df = all_df[["text", "class", "tokens"]].dropna().copy()
df["text"] = df["text"].astype(str)
df["class"] = df["class"].astype(str)

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["class"],
)
y_train = train_df["class"]
y_test = test_df["class"]

class_probs = y_train.value_counts(normalize=True)
labels = class_probs.index.to_list()
probs = class_probs.values

rng = np.random.default_rng(42)
y_pred = rng.choice(labels, size=len(y_test), p=probs)

print("\nConfusion matrix:")
print(pd.crosstab(y_test, y_pred, rownames=['True'], colnames=['Predicted']))
print("Baseline Test 1 - random label Classification Report:")
print(classification_report(y_test, y_pred, labels=labels))


Confusion matrix:
Predicted  notsarc  sarc
True                    
notsarc        468   471
sarc           477   462
Baseline Test 1 - random label Classification Report:
              precision    recall  f1-score   support

     notsarc       0.50      0.50      0.50       939
        sarc       0.50      0.49      0.49       939

    accuracy                           0.50      1878
   macro avg       0.50      0.50      0.50      1878
weighted avg       0.50      0.50      0.50      1878



In [ ]:
# Baseline 2: TF-IDF + SVM
df_tfidf = all_df.dropna(subset=['tokens', 'class']).copy()
X = df_tfidf['tokens']
y = df_tfidf['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

def dummy_fn(doc):
    return doc

tfidf = TfidfVectorizer(
    analyzer = 'word',
    tokenizer = dummy_fn,
    preprocessor = dummy_fn,
    token_pattern = None,
    max_features = 10000
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

svm_model = LinearSVC(random_state = 42)
svm_model.fit(X_train_tfidf, y_train)
y_pred = svm_model.predict(X_test_tfidf)
display(pd.crosstab(y_test, y_pred, rownames=['True'], colnames=['Predicted']))

print("Baseline Test 2 TF-IDF + SVM Classification Report:")
print(classification_report(y_test, y_pred))

Predicted,notsarc,sarc
True,,
notsarc,716,223
sarc,257,682


Baseline Test 2 TF-IDF + SVM Classification Report:
              precision    recall  f1-score   support

     notsarc       0.74      0.76      0.75       939
        sarc       0.75      0.73      0.74       939

    accuracy                           0.74      1878
   macro avg       0.74      0.74      0.74      1878
weighted avg       0.74      0.74      0.74      1878



## Set up and VADER

In [9]:
# Label mapping
# notsarc-> 0, sarc-> 1
label_map = {'notsarc': 0, 'sarc': 1}
train_df['label_idx'] = train_df['class'].map(label_map)
test_df['label_idx'] = test_df['class'].map(label_map)

all_train_tokens = [token for sublist in train_df['tokens'].tolist() for token in sublist]
unique_tokens = list(set(all_train_tokens))

word2idx = {word: idx + 2 for idx, word in enumerate(unique_tokens)}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1
VOCAB_SIZE = len(word2idx)

print(f"Vocabulary Size: {VOCAB_SIZE}")

Vocabulary Size: 24782


In [10]:
# VADER socres for all tokens
def get_vader_sequence(tokens):
    # Returns a list of [pos, neu, neg] for each token
    seq = []
    for t in tokens:
        scores = sia.polarity_scores(t)
        seq.append([scores['pos'], scores['neu'], scores['neg']])
    return seq

train_df['vader_feats'] = train_df['tokens'].apply(get_vader_sequence)
test_df['vader_feats'] = test_df['tokens'].apply(get_vader_sequence)

## PyTorch Dataset and DataLoader

In [11]:
MAX_LEN = 60

class SarcasmDualChannelDataset(Dataset):
    def __init__(self, df, word2idx, max_len):
        self.tokens = df['tokens'].tolist()
        self.vader_feats = df['vader_feats'].tolist()
        self.labels = df['label_idx'].tolist()
        self.word2idx = word2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        tokens = self.tokens[idx]
        vader_seq = self.vader_feats[idx]
        
        # Channel A: Semantic ID Mapping
        token_ids = [self.word2idx.get(t, self.word2idx["<UNK>"]) for t in tokens]
        
        # Truncate or Pad
        if len(token_ids) >= self.max_len:
            token_ids = token_ids[:self.max_len]
            vader_seq = vader_seq[:self.max_len]
        else:
            pad_len = self.max_len - len(token_ids)
            token_ids = token_ids + [self.word2idx["<PAD>"]] * pad_len
            vader_seq = vader_seq + [[0.0, 0.0, 0.0]] * pad_len
            
        return {
            'semantic_ids': torch.tensor(token_ids, dtype=torch.long),
            'sentiment_feat': torch.tensor(vader_seq, dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

BATCH_SIZE = 128

train_dataset = SarcasmDualChannelDataset(train_df, word2idx, MAX_LEN)
test_dataset = SarcasmDualChannelDataset(test_df, word2idx, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## DMS-CNN model

channel b - VADER

2,3,5 滑动窗口

在dimension = 1 就直接暴力拼接。

feature regularization X

In [ ]:
class DMSCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, num_filters=64, filter_sizes=[2, 3, 5]):
        super(DMSCNN, self).__init__()
        
        # Dual Channel Embedding
        # Channel A: Semantic
        self.semantic_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # Channel B: Sentiment
        sentiment_dim = 3  
        
        # Multi-Scale Convolution
        self.convs_semantic = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=k, padding='same')
            for k in filter_sizes
        ])
        
        self.convs_sentiment = nn.ModuleList([
            nn.Conv1d(in_channels=sentiment_dim, out_channels=num_filters, kernel_size=k, padding='same')
            for k in filter_sizes
        ])
        
        # Attention
        total_filters = num_filters * len(filter_sizes) * 2
        self.attention_linear = nn.Linear(total_filters, 1)
        
        # Classification
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(total_filters, 1)
    
    def forward(self, semantic_ids, sentiment_feat):
        # Embedding
        # semantic_ids shape: (batch, seq_len) -> (batch, seq_len, embed_dim) -> permute -> (batch, embed_dim, seq_len)
        sem_x = self.semantic_embedding(semantic_ids).permute(0, 2, 1) 
        
        # sentiment_feat shape: (batch, seq_len, 3) -> permute -> (batch, 3, seq_len)
        sent_x = sentiment_feat.permute(0, 2, 1)
        
        # Multi-Scale Convolutions
        # ReLU activation
        sem_conv_outs = [F.relu(conv(sem_x)) for conv in self.convs_semantic]
        sent_conv_outs = [F.relu(conv(sent_x)) for conv in self.convs_sentiment]
        
        # Feature Concatenation
        # Shape becomes (batch, total_filters, seq_len)
        f_combined = torch.cat(sem_conv_outs + sent_conv_outs, dim=1)
        
        # Permute back to (batch, seq_len, total_filters) for Attention calculation
        f_combined = f_combined.permute(0, 2, 1)
        
        # Self-Attention
        # attn_scores shape: (batch, seq_len, 1)
        attn_scores = self.attention_linear(f_combined)
        attn_weights = torch.softmax(attn_scores, dim=1)
        
        # Context Vector: Weighted sum of features across the sequence length
        # context_vector shape: (batch, total_filters)
        context_vector = torch.sum(attn_weights * f_combined, dim=1)
        
        # Classification
        context_vector = self.dropout(context_vector)
        output_prob = torch.sigmoid(self.classifier(context_vector))
        
        return output_prob.squeeze(-1)

## Model Train

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = DMSCNN(vocab_size=VOCAB_SIZE).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 10

wandb.init(
    project="SI630-Project-Sarcasm-Detection",
    name="DMS-CNN-Final",
    config={
        "architecture": "DMS-CNN",
        "epochs": EPOCHS,
        "learning_rate": 0.001,
        "weight_decay": 0.00, # do not delete
        "vocab_size": VOCAB_SIZE
    }
)

current_max_f1 = 0.0
best_model_weights = copy.deepcopy(model.state_dict())

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct_train = 0
    total_train = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    
    for batch in progress_bar:
        sem_ids = batch['semantic_ids'].to(device)
        sent_feat = batch['sentiment_feat'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        predictions = model(sem_ids, sent_feat)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds_class = (predictions > 0.5).float()
        correct_train += (preds_class == labels).sum().item()
        total_train += labels.size(0)
        
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
        wandb.log({"batch_train_loss": loss.item()})
        
    epoch_loss = total_loss / len(train_loader)
    train_acc = correct_train / total_train
    
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in test_loader:
            sem_ids = batch['semantic_ids'].to(device)
            sent_feat = batch['sentiment_feat'].to(device)
            labels = batch['label'].to(device)
            
            probs = model(sem_ids, sent_feat)
            preds_class = (probs > 0.5).float()
            
            all_preds.extend(preds_class.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    val_acc = accuracy_score(all_targets, all_preds)
    val_f1 = f1_score(all_targets, all_preds, pos_label=1)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {epoch_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
    
    wandb.log({
        "epoch": epoch + 1,
        "epoch_train_loss": epoch_loss,
        "epoch_train_acc": train_acc,
        "epoch_val_acc": val_acc,
        "epoch_val_f1": val_f1
    })
    
    # Save the best model
    if val_f1 > current_max_f1:
        current_max_f1 = val_f1
        best_model_weights = copy.deepcopy(model.state_dict())

model.load_state_dict(best_model_weights)

batch_train_loss,▇▇▇▆▅▅▆▆▅▃▅▃▄▅█▄▄▅▃▄▃▆▅▄▃▄▄▁▄▄▂▃▂▃▄▅▂▂▂▅
epoch,▁▂▃▄▅▆▇█
epoch_train_acc,▁▅▆▇▇▇██
epoch_train_loss,█▅▄▃▃▂▂▁
epoch_val_acc,▁▅▆█▆█▇█
epoch_val_f1,▁▇▇██▇█▇
batch_train_loss,0.5454
epoch,8
epoch_train_acc,0.68487
epoch_train_loss,0.59012
epoch_val_acc,0.68903


Epoch 1/10 | Train Loss: 0.6546 | Train Acc: 0.6281 | Val Acc: 0.6731 | Val F1: 0.6748


Epoch 2/10 | Train Loss: 0.6052 | Train Acc: 0.6751 | Val Acc: 0.6821 | Val F1: 0.7019


Epoch 3/10 | Train Loss: 0.5592 | Train Acc: 0.7174 | Val Acc: 0.6949 | Val F1: 0.6772


Epoch 4/10 | Train Loss: 0.4968 | Train Acc: 0.7623 | Val Acc: 0.7018 | Val F1: 0.6960


...

Epoch 7/10 | Train Loss: 0.2476 | Train Acc: 0.8993 | Val Acc: 0.7061 | Val F1: 0.6964


Epoch 8/10 | Train Loss: 0.1715 | Train Acc: 0.9349 | Val Acc: 0.6991 | Val F1: 0.7003


Epoch 9/10 | Train Loss: 0.1163 | Train Acc: 0.9572 | Val Acc: 0.6912 | Val F1: 0.6982


Epoch 10/10 | Train Loss: 0.0686 | Train Acc: 0.9760 | Val Acc: 0.6848 | Val F1: 0.6790


<All keys matched successfully>

In [18]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Final Evaluation"):
        sem_ids = batch['semantic_ids'].to(device)
        sent_feat = batch['sentiment_feat'].to(device)
        labels = batch['label'].to(device)
        
        probs = model(sem_ids, sent_feat)
        preds_class = (probs > 0.5).float()
        
        all_preds.extend(preds_class.cpu().numpy())
        all_targets.extend(labels.cpu().numpy())

# Calculate Metrics
from sklearn.metrics import classification_report

final_test_acc = accuracy_score(all_targets, all_preds)
final_test_f1 = f1_score(all_targets, all_preds, pos_label=1)

print(f"\nDMS-CNN Final Test Accuracy: {final_test_acc:.4f}")
print(f"DMS-CNN Final F1-Score (sarc): {final_test_f1:.4f}")
print("\nClassification Report:")
print(classification_report(all_targets, all_preds, target_names=['notsarc', 'sarc']))

wandb.log({
    "final_test_accuracy": final_test_acc,
    "final_test_f1": final_test_f1
})

wandb.finish()

Final Evaluation: 100%|██████████| 15/15 [00:01<00:00, 14.21it/s]



DMS-CNN Final Test Accuracy: 0.7018
DMS-CNN Final F1-Score (sarc): 0.7192

Classification Report:
              precision    recall  f1-score   support

     notsarc       0.73      0.64      0.68       939
        sarc       0.68      0.76      0.72       939

    accuracy                           0.70      1878
   macro avg       0.70      0.70      0.70      1878
weighted avg       0.70      0.70      0.70      1878



batch_train_loss,████▇▇▇▇▇▆▆▆▆▇▅▆▄▄▄▅▃▄▃▅▃▃▄▃▃▂▂▂▁▂▁▂▁▁▁▁
epoch,▁▂▃▃▄▅▆▆▇█
epoch_train_acc,▁▂▃▄▅▆▆▇██
epoch_train_loss,█▇▇▆▅▄▃▂▂▁
epoch_val_acc,▁▃▆▇▇▆█▇▅▃
epoch_val_f1,▁▅▁▄█▆▄▅▅▂
final_test_accuracy,▁
final_test_f1,▁
batch_train_loss,0.05408
epoch,10
epoch_train_acc,0.97603


## BERT-based Sarcasm Classification


In [7]:
label2id = {"notsarc": 0, "sarc": 1}
id2label = {0: "notsarc", 1: "sarc"}


def prepare_text_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["text"] = out["text"].fillna("").astype(str)
    out["label"] = out["class"].map(label2id)
    out = out.dropna(subset=["label"]).reset_index(drop=True)
    out["label"] = out["label"].astype(int)
    return out[["text", "label"]]


train_text = prepare_text_df(train_df)
test_text = prepare_text_df(test_df)

set_seed(42)

In [8]:
# constants
model_name = "bert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 8
num_epochs = 2
learning_rate = 2e-5
SAVE_DIR = Path("./bert_sarcasm_finetuned")

tokenizer_bert = BertTokenizerFast.from_pretrained(model_name)

class TextClsDataset(torch.utils.data.Dataset):
    def __init__(self, df: pd.DataFrame):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer_bert(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LEN,
        )
        enc["labels"] = int(self.labels[idx])
        return enc

train_ds = TextClsDataset(train_text)
test_ds = TextClsDataset(test_text)

def compute_metrics(eval_pred):
    preds = eval_pred.predictions
    labels = eval_pred.label_ids

    if isinstance(preds, (tuple, list)):
        preds = preds[0]

    preds = np.asarray(preds)
    y_pred = np.argmax(preds, axis=1)

    return {
        "accuracy": accuracy_score(labels, y_pred),
        "f1_sarc": f1_score(labels, y_pred, pos_label=1),
    }


In [11]:
wandb.init(
    project="SI630-Project-Sarcasm-Detection",
    name="BERT-finetune",
    config={
        "model": model_name,
        "max_len": MAX_LEN,
        "epochs": num_epochs,
        "lr": learning_rate,
        "train_bs": 8,
        "eval_bs": 16,
        "load_if_exists": True,
        "save_dir": str(SAVE_DIR),
    },
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer_bert, padding="longest")
if SAVE_DIR.exists() and (SAVE_DIR / "config.json").exists():
    print(f"Found saved model & skipping training.")
    model_bert = BertForSequenceClassification.from_pretrained(SAVE_DIR)
    tokenizer_bert = BertTokenizerFast.from_pretrained(SAVE_DIR)
else:
    print("No saved model found — starting training.")

    model_bert = BertForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
    )

    tokenizer_bert = BertTokenizerFast.from_pretrained(model_name)

    training_args = TrainingArguments(
        output_dir="./bert_sarcasm_ckpt",
        num_train_epochs=num_epochs,
        learning_rate=learning_rate,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        warmup_ratio=0.06,
        weight_decay=0.01,
        logging_steps=25,
        save_strategy="no",
        report_to=["wandb"],
    )

    trainer = Trainer(
        model=model_bert,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # Save
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(SAVE_DIR)
    tokenizer_bert.save_pretrained(SAVE_DIR)
    print(f"Saved fine-tuned BERT to: {SAVE_DIR.resolve()}")

wandb.finish()

Found saved model & skipping training.


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2104.29it/s]


In [12]:
test_dl = DataLoader(test_ds, batch_size=16, shuffle=False, collate_fn=data_collator)

model_bert.eval()
all_preds = []
all_true = []

with torch.no_grad():
    for batch in test_dl:
        labels = batch.pop('labels')
        batch = {k: v.to(model_bert.device) for k, v in batch.items()}

        logits = model_bert(**batch).logits  # (B, 2)
        preds = torch.argmax(logits, dim=1).cpu().numpy()

        all_preds.extend(preds.tolist())
        all_true.extend(labels.numpy().tolist())

print("\nClassification report:")
print(classification_report(all_true, all_preds, target_names=["notsarc", "sarc"]))


Classification report:
              precision    recall  f1-score   support

     notsarc       0.90      0.91      0.90       939
        sarc       0.91      0.90      0.90       939

    accuracy                           0.90      1878
   macro avg       0.90      0.90      0.90      1878
weighted avg       0.90      0.90      0.90      1878



# Bert for feature extraction(semantics) and CNN for classifiction

In [ ]:
max_len_hybrid = 128
batch_size_hybrid = 32

bert_tokenizer = BertTokenizerFast.from_pretrained(model_name)

def _prepare_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["text"] = out["text"].fillna("").astype(str)
    # ensure tokens exist
    if "tokens" not in out.columns:
        out["tokens"] = out["text"].apply(tokenizer.tokenize)
    # ensure numeric labels exist
    if "label_idx" not in out.columns:
        out["label_idx"] = out["class"].astype(str).map(label2id)
    out = out.dropna(subset=["label_idx"]).reset_index(drop=True)
    out["label_idx"] = out["label_idx"].astype(int)
    return out[["text", "tokens", "label_idx"]]

train_hybrid_df = _prepare_df(train_df)
test_hybrid_df = _prepare_df(test_df)


def vader_seq_from_tokens(tokens, max_len: int):
    feats = []
    for t in tokens[:max_len]:
        s = sia.polarity_scores(t)
        feats.append([s["pos"], s["neu"], s["neg"]])
    if len(feats) < max_len:
        feats.extend([[0.0, 0.0, 0.0]] * (max_len - len(feats)))
    return feats


class HybridBertVaderDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.texts = df["text"].tolist()
        self.tokens = df["tokens"].tolist()
        self.labels = df["label_idx"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx: int):
        text = self.texts[idx]
        toks = self.tokens[idx]
        label = int(self.labels[idx])

        enc = bert_tokenizer(
            text,
            truncation=True,
            max_length=max_len_hybrid,
            padding="max_length",
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)

        vader_feats = torch.tensor(
            vader_seq_from_tokens(toks, max_len_hybrid),
            dtype=torch.float32,
        )  # (L, 3)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "vader_feats": vader_feats,
            "labels": torch.tensor(label, dtype=torch.long),
        }


train_hybrid_ds = HybridBertVaderDataset(train_hybrid_df)
test_hybrid_ds = HybridBertVaderDataset(test_hybrid_df)

train_hybrid_loader = DataLoader(train_hybrid_ds, batch_size=batch_size_hybrid, shuffle=True)
test_hybrid_loader = DataLoader(test_hybrid_ds, batch_size=batch_size_hybrid, shuffle=False)

In [17]:
class BertVaderDMSCNN(nn.Module):
    def __init__(self, model_name=model_name, num_filters=64, filter_sizes=(2, 3, 5), dropout=0.3, freeze_bert=True):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        hidden = self.bert.config.hidden_size

        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False

        # Semantic channel CNNs - for classification
        self.convs_sem = nn.ModuleList([
            nn.Conv1d(in_channels=hidden, out_channels=num_filters, kernel_size=k, padding="same")
            for k in filter_sizes
        ])

        # Sentiment channel CNNs
        self.convs_sent = nn.ModuleList([
            nn.Conv1d(in_channels=3, out_channels=num_filters, kernel_size=k, padding="same")
            for k in filter_sizes
        ])

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(num_filters * len(filter_sizes) * 2, 2)

    def forward(self, input_ids, attention_mask, vader_feats):
        # BERT embeddings: (B, L, H)
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        h = out.last_hidden_state

        # Semantic conv expects (B, H, L)
        h_sem = h.permute(0, 2, 1)
        sem_feats = [F.relu(conv(h_sem)) for conv in self.convs_sem]

        # Sentiment conv expects (B, 3, L)
        h_sent = vader_feats.permute(0, 2, 1)
        sent_feats = [F.relu(conv(h_sent)) for conv in self.convs_sent]

        # Global max pool over sequence length
        sem_pooled = [torch.max(f, dim=2).values for f in sem_feats]
        sent_pooled = [torch.max(f, dim=2).values for f in sent_feats]

        feats = torch.cat(sem_pooled + sent_pooled, dim=1)
        feats = self.dropout(feats)
        logits = self.classifier(feats)
        return logits

hybrid_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

hybrid_model = BertVaderDMSCNN(freeze_bert=True).to(hybrid_device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam([p for p in hybrid_model.parameters() if p.requires_grad], lr=1e-3)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2563.50it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:

SAVE_DIR = Path("./hybrid_bert_vader_cnn")
ckpt_path = SAVE_DIR / "pytorch_model.bin"
meta_path = SAVE_DIR / "metadata.json"

epoch_hybrid = 8

wandb.init(
    project="SI630-Project-Sarcasm-Detection",
    name="Hybrid-BERT(VaderCNN)",
    config={
        "bert": globals().get("model_name", "bert-base-uncased"),
        "max_len": max_len_hybrid,
        "epochs": epoch_hybrid,
        "batch_size": batch_size_hybrid,
        "lr": 1e-5,
        "load_if_exists": True,
        "save_dir": str(SAVE_DIR),
    },
)

def run_eval(model, loader):
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(hybrid_device)
            attention_mask = batch["attention_mask"].to(hybrid_device)
            vader_feats = batch["vader_feats"].to(hybrid_device)
            labels = batch["labels"].to(hybrid_device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask, vader_feats=vader_feats)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_true.extend(labels.cpu().numpy().tolist())

    acc = accuracy_score(all_true, all_preds)
    f1 = f1_score(all_true, all_preds, pos_label=1)
    return acc, f1

if ckpt_path.exists():
    print(f"Load saved hybrid & skipping training.")
    hybrid_model.load_state_dict(torch.load(ckpt_path, map_location=hybrid_device))
    hybrid_model.eval()
else:
    print("No saved hybrid checkpoint found — starting training.")

    global_step = 0
    for epoch in range(epoch_hybrid):
        hybrid_model.train()
        total_loss = 0.0

        pbar = tqdm(train_hybrid_loader, desc=f"Hybrid Epoch {epoch+1}/{epoch_hybrid}", leave=False)
        for batch in pbar:
            input_ids = batch["input_ids"].to(hybrid_device)
            attention_mask = batch["attention_mask"].to(hybrid_device)
            vader_feats = batch["vader_feats"].to(hybrid_device)
            labels = batch["labels"].to(hybrid_device)

            optimizer.zero_grad()
            logits = hybrid_model(input_ids=input_ids, attention_mask=attention_mask, vader_feats=vader_feats)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

            wandb.log({"train/loss": loss.item(), "step": global_step, "epoch": epoch + 1})
            global_step += 1

        avg_loss = total_loss / max(1, len(train_hybrid_loader))
        acc, f1 = run_eval(hybrid_model, test_hybrid_loader)
        wandb.log({"epoch/loss": avg_loss, "eval/acc": acc, "eval/f1_sarc": f1, "epoch": epoch + 1})
        print(f"Epoch {epoch+1}: loss={avg_loss:.4f} | test_acc={acc:.4f} | test_f1(sarc)={f1:.4f}")

    # Save
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    torch.save(hybrid_model.state_dict(), ckpt_path)
    metadata = {
        "model_name": globals().get("model_name", "bert-base-uncased"),
        "max_len_hybrid": int(globals().get("max_len_hybrid", 128)),
        "num_labels": 2,
    }
    meta_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print(f"Saved hybrid model")

# Final report
hybrid_model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for batch in test_hybrid_loader:
        input_ids = batch["input_ids"].to(hybrid_device)
        attention_mask = batch["attention_mask"].to(hybrid_device)
        vader_feats = batch["vader_feats"].to(hybrid_device)
        labels = batch["labels"].to(hybrid_device)

        logits = hybrid_model(input_ids=input_ids, attention_mask=attention_mask, vader_feats=vader_feats)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_true.extend(labels.cpu().numpy().tolist())

print(classification_report(all_true, all_preds, target_names=["notsarc", "sarc"]))

wandb.finish()

Load saved hybrid & skipping training.
              precision    recall  f1-score   support

     notsarc       0.85      0.86      0.85       939
        sarc       0.86      0.84      0.85       939

    accuracy                           0.85      1878
   macro avg       0.85      0.85      0.85      1878
weighted avg       0.85      0.85      0.85      1878

